# Chatterbox Egyptian voice test (gesture predictions)

Standalone notebook for **Laghtna Chatterbox** only (`laghtna-chatterbox-v1 model/`).
Gesture predictions use the saved `.pkl` bundle **before** the Chatterbox pip bootstrap (NumPy 2.x for pickles). TTS text is **mapped pronunciation + `.`** only.

**Reference (read-only):** `CombinedDatasetModel.ipynb` glove demo inputs.

## 1. Paths + gesture predictions (run **before** Chatterbox bootstrap)

Loads `CombinedGesture_model.pkl`, `Modelscaler.pkl`, `label_encoder.pkl` with NumPy 2.x and caches results to `generated_audio/chatterbox_gesture_cache.json`.

In [1]:
import json
import time
import traceback
from pathlib import Path

import joblib
import numpy as np

try:
    import pandas as pd
except ImportError:
    pd = None

BASE_DIR = Path.cwd().resolve()
print("cwd:", BASE_DIR)

FEATURE_COLS = [
    "flex1", "flex2", "flex3", "flex4", "flex5",
    "accX", "accY", "accZ",
    "gyroX", "gyroY", "gyroZ",
]

CHATTERBOX_DIR = BASE_DIR / "laghtna-chatterbox-v1 model"
AUDIO_ROOT = BASE_DIR / "generated_audio"
CHATTER_AUDIO_DIR = AUDIO_ROOT / "chatterbox"
CACHE_DIR = AUDIO_ROOT / "_cache"
GESTURE_CACHE_JSON = AUDIO_ROOT / "chatterbox_gesture_cache.json"

for d in (AUDIO_ROOT, CHATTER_AUDIO_DIR, CACHE_DIR):
    d.mkdir(parents=True, exist_ok=True)

EXPECTED_CB_FILES = [
    "conds.pt",
    "mtl_tokenizer.json",
    "s3gen.safetensors",
    "t3_cfg.safetensors",
    "t3_mtl23ls_v2.safetensors",
    "tokenizer.json",
    "ve.safetensors",
]

print("\nChatterbox weight files:")
for name in EXPECTED_CB_FILES:
    p = CHATTERBOX_DIR / name
    print(f"  [{'OK' if p.is_file() else 'MISS'}] {name}")

print("\nNumPy (gesture pickles need 2.x):", np.__version__)

# --- Glove cases copied from CombinedDatasetModel.ipynb (cells 16–18); not edited ---
REAL_GLOVES_CASES = [
    {
        "name": "GlovesInput_demo_أ",
        "expected": "أ",
        "values": [865, 472, 632, 593, 675, -8380, 14150, 410, -610, -1310, 275],
    },
    {
        "name": "GlovesInput_demo_اثنين",
        "expected": "اثنين",
        "values": [780, 400, 645, 782, 946, -8400, 14164, -1900, -100, -1500, -400],
    },
    {
        "name": "GlovesInput_demo_خمسة",
        "expected": "خمسة",
        "values": [910, 885, 945, 805, 960, 1700, 16450, 450, 100, -150, -400],
    },
]

import subprocess
import sys

# Run gesture inference in a child process with NumPy 2.x (pickles use PCG64).
# Parent env may later downgrade NumPy for Chatterbox — cache survives on disk.
_GESTURE_PRED_SCRIPT = (
    "import json, time, traceback\n"
    "from pathlib import Path\n"
    "import subprocess, sys\n"
    "subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',\n"
    "    'numpy>=2.1,<2.2', 'joblib', 'scikit-learn', 'pandas'])\n"
    "import joblib\n"
    "import numpy as np\n"
    "try:\n    import pandas as pd\nexcept ImportError:\n    pd = None\n"
    f"BASE = Path({str(BASE_DIR)!r})\n"
    f"CASES = json.loads({json.dumps(REAL_GLOVES_CASES, ensure_ascii=False)!r})\n"
    "OUT = BASE / 'generated_audio' / 'chatterbox_gesture_cache.json'\n"
    "clf = scaler = label_encoder = None\n"
    "err = None\n"
    "results = []\n"
    "t0 = time.perf_counter()\n"
    "try:\n"
    "    clf = joblib.load(BASE / 'CombinedGesture_model.pkl')\n"
    "    scaler = joblib.load(BASE / 'Modelscaler.pkl')\n"
    "    label_encoder = joblib.load(BASE / 'label_encoder.pkl')\n"
    "    for i, case in enumerate(CASES, start=1):\n"
    "        row = np.array(case['values'], dtype=np.float64).reshape(1, -1)\n"
    "        fn = getattr(scaler, 'feature_names_in_', None)\n"
    "        X_in = pd.DataFrame(row, columns=list(fn)) if fn is not None and pd is not None else row\n"
    "        t_inf0 = time.perf_counter()\n"
    "        enc = clf.predict(scaler.transform(X_in))\n"
    "        infer_s = time.perf_counter() - t_inf0\n"
    "        pred = str(label_encoder.inverse_transform(enc)[0])\n"
    "        results.append(dict(test=i, slug='real_glove_' + str(i), name=case.get('name'),\n"
    "            expected=case.get('expected'), values=case.get('values'),\n"
    "            encoded_index=int(enc[0]), predicted=pred, infer_s=infer_s))\n"
    "except Exception:\n"
    "    err = traceback.format_exc()\n"
    "load_s = time.perf_counter() - t0\n"
    "OUT.parent.mkdir(parents=True, exist_ok=True)\n"
    "with open(OUT, 'w', encoding='utf-8') as fh:\n"
    "    json.dump({'real_glove_results': results, 'gesture_load_error': err,\n"
    "        'gesture_load_s': load_s, 'numpy_version': np.__version__},\n"
    "        fh, ensure_ascii=False, indent=2)\n"
    "print('gesture subprocess ok | preds:', len(results), '| numpy:', np.__version__)\n"
)

print("Running gesture predictions (subprocess, NumPy 2.x)...")
_tg0 = time.perf_counter()
_proc = subprocess.run(
    [sys.executable, "-c", _GESTURE_PRED_SCRIPT],
    cwd=str(BASE_DIR),
    capture_output=True,
    text=True,
)
print(_proc.stdout or "")
if _proc.stderr:
    print(_proc.stderr[-2000:])
if _proc.returncode != 0:
    print("Gesture subprocess exit code:", _proc.returncode)
GESTURE_LOAD_S = time.perf_counter() - _tg0
print(f"Gesture subprocess wall time: {GESTURE_LOAD_S:.4f}s")

with open(GESTURE_CACHE_JSON, encoding="utf-8") as fh:
    _gc = json.load(fh)

GESTURE_LOAD_ERROR = _gc.get("gesture_load_error")
REAL_GLOVE_RESULTS = _gc.get("real_glove_results", [])
if GESTURE_LOAD_ERROR:
    print("Gesture load error (subprocess):\n", GESTURE_LOAD_ERROR[-1200:])
else:
    print("\nReal glove predictions:", [(r["expected"], r["predicted"]) for r in REAL_GLOVE_RESULTS])
print("Cache:", GESTURE_CACHE_JSON.resolve())


cwd: C:\Users\sondo\Desktop\Voice Model Stuff

Chatterbox weight files:
  [OK] conds.pt
  [OK] mtl_tokenizer.json
  [OK] s3gen.safetensors
  [OK] t3_cfg.safetensors
  [OK] t3_mtl23ls_v2.safetensors
  [OK] tokenizer.json
  [OK] ve.safetensors

NumPy (gesture pickles need 2.x): 2.1.3
Running gesture predictions (subprocess, NumPy 2.x)...


gesture subprocess ok | preds: 3 | numpy: 2.1.3


[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: C:\Users\sondo\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip

Gesture subprocess wall time: 27.7867s

Real glove predictions: [('أ', 'أ'), ('اثنين', 'اثنين'), ('خمسة', 'خمسة')]
Cache: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\chatterbox_gesture_cache.json


## 2. Chatterbox environment bootstrap (`run-once`)

Installs **chatterbox-tts** + **transformers 5.x** + **NumPy &lt; 2** (Chatterbox stack). **Restart kernel** after first run if imports fail, then set `CB_NB_SKIP_BOOTSTRAP=1` and Run All.

In [2]:
import os
import subprocess
import sys

if os.environ.get("CB_NB_SKIP_BOOTSTRAP") == "1":
    print("[bootstrap] skipped (CB_NB_SKIP_BOOTSTRAP=1)")
else:
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "perth"], check=False)
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "setuptools<81",
            "resemble-perth",
            "chatterbox-tts",
            "huggingface_hub",
            "soundfile",
            "torch",
            "torchaudio",
            "ipython",
            "nbformat",
        ]
    )
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "--force-reinstall",
            "transformers==5.2.0",
            "safetensors==0.5.3",
            "numpy>=1.24,<2.0",
        ]
    )
    print(
        "\nBootstrap done. If `import chatterbox` fails, restart kernel, set CB_NB_SKIP_BOOTSTRAP=1, Run All."
    )


[bootstrap] skipped (CB_NB_SKIP_BOOTSTRAP=1)


## 3. Device + dependency probe

In [3]:
import traceback

HAVE_CHATTERBOX = False
HAVE_SAFE = False
HAVE_HF = False
HAVE_SF = False
torch = None
CUDA_AVAILABLE = False
DEVICE_PREF = "cpu"
GPU_NAME = None

try:
    import torch

    CUDA_AVAILABLE = torch.cuda.is_available()
    DEVICE_PREF = "cuda" if CUDA_AVAILABLE else "cpu"
    if CUDA_AVAILABLE:
        GPU_NAME = torch.cuda.get_device_name(0)
except ImportError:
    print("torch not installed — pip install torch")

print("CUDA available:", CUDA_AVAILABLE)
print("GPU name:", GPU_NAME or "(n/a)")
print("Selected device:", DEVICE_PREF)

for mod, pip in [
    ("safetensors", "safetensors==0.5.3"),
    ("huggingface_hub", "huggingface_hub"),
    ("soundfile", "soundfile"),
]:
    try:
        __import__(mod)
        print("[deps OK]", mod)
        if mod == "safetensors":
            HAVE_SAFE = True
        if mod == "huggingface_hub":
            HAVE_HF = True
        if mod == "soundfile":
            HAVE_SF = True
    except ImportError as e:
        print("[deps MISSING]", mod, "-> pip install", pip, "|", e)

try:
    import chatterbox  # noqa: F401

    HAVE_CHATTERBOX = True
    print("[deps OK] chatterbox")
except ImportError:
    print("[deps MISSING] chatterbox -> pip install chatterbox-tts")
    traceback.print_exc()


CUDA available: False
GPU name: (n/a)
Selected device: cpu
[deps OK] safetensors
[deps OK] huggingface_hub
[deps OK] soundfile


[deps OK] chatterbox


## 4. Egyptian pronunciation mapping

In [4]:
EG_LETTER = {
    "أ": "أَلِف", "ا": "أَلِف", "ب": "بِه", "ت": "تِه", "ث": "سِه",
    "ج": "جِيم", "ح": "حَا", "خ": "خَا", "د": "دَال", "ذ": "ذَال",
    "ر": "رَا", "ز": "زَاي", "س": "سِين", "ش": "شِين", "ص": "صَاد",
    "ض": "ضَاد", "ط": "طَا", "ظ": "ظَا", "ع": "عِين", "غ": "غِين",
    "ف": "فِه", "ق": "أَاف", "ك": "كَاف", "ل": "لَام", "م": "مِيم",
    "ن": "نُون", "ه": "هَا", "و": "وَاو", "ي": "يَا",
}
DIGIT_MAP = {
    "0": "صِفْر", "1": "وَاحِد", "2": "اِتْنِين", "3": "تَلَاتَه", "4": "أَرْبَعَه",
    "5": "خَمْسَه", "6": "سِتَّه", "7": "سَبْعَه", "8": "تَمَانْيَه", "9": "تِسْعَه",
}
AR_WORD_NUM = {
    "صفر": DIGIT_MAP["0"], "واحد": DIGIT_MAP["1"], "اثنين": DIGIT_MAP["2"],
    "ثلاثة": DIGIT_MAP["3"], "أربعة": DIGIT_MAP["4"], "خمسة": DIGIT_MAP["5"],
    "ستة": DIGIT_MAP["6"], "سبعة": DIGIT_MAP["7"], "ثمانية": DIGIT_MAP["8"],
    "تسعة": DIGIT_MAP["9"], "عشرة": "عَشَرَة",
}
EG_WORD_TTS = {
    "بيت": "بَيْت", "باب": "بَاب", "ماما": "مَامَا",
    "خمسة": AR_WORD_NUM["خمسة"], "اثنين": AR_WORD_NUM["اثنين"],
}
EG_SENTENCE_TTS = {
    "انا بيت": "أَنا بَيْت",
    "أنا بيت": "أَنا بَيْت",
    "أنا بسمعك": "أَنَا بَسْمَعَك",
    "انا بسمعك": "أَنَا بَسْمَعَك",
}


def normalize_pred_token(pred) -> str:
    return str(pred).strip() if pred is not None else ""


def map_prediction_to_egyptian_pronunciation(pred) -> str:
    tok = normalize_pred_token(pred)
    if tok in EG_LETTER:
        return EG_LETTER[tok]
    if tok in AR_WORD_NUM:
        return AR_WORD_NUM[tok]
    if tok in DIGIT_MAP:
        return DIGIT_MAP[tok]
    if tok in ("آ", "إ", "ٱ"):
        return EG_LETTER.get("أ", tok)
    return tok


def finalize_chatterbox_utterance(s: str) -> str:
    t = (s or "").strip()
    if not t:
        return t
    return t if t.endswith(".") else t + "."


def speak_single_prediction(pred) -> str:
    return finalize_chatterbox_utterance(map_prediction_to_egyptian_pronunciation(pred))


def speak_predictions_as_letters(predictions) -> str:
    toks = [normalize_pred_token(x) for x in predictions if normalize_pred_token(x)]
    if not toks:
        return ""
    parts = [map_prediction_to_egyptian_pronunciation(t) for t in toks]
    return finalize_chatterbox_utterance("، ".join(parts[:50]))


def speak_predictions_as_joined_word(predictions) -> str:
    toks = [normalize_pred_token(x) for x in predictions if normalize_pred_token(x)]
    surf = "".join(toks[:50])
    if not surf:
        return ""
    inner = EG_WORD_TTS[surf] if surf in EG_WORD_TTS else surf
    return finalize_chatterbox_utterance(inner)


def speak_predictions_as_sentence(predictions) -> str:
    pieces = []
    for x in predictions:
        if x is None:
            continue
        if isinstance(x, str) and x.strip() == "" and len(x) > 0:
            pieces.append(" ")
            continue
        t = normalize_pred_token(x)
        if not t:
            continue
        pieces.append(" " if t.isspace() else t)
    joined = " ".join("".join(pieces).split())
    if not joined:
        return ""
    if joined in EG_SENTENCE_TTS:
        return finalize_chatterbox_utterance(EG_SENTENCE_TTS[joined])
    return finalize_chatterbox_utterance(
        " ".join(EG_WORD_TTS.get(w, w) if w in EG_WORD_TTS else map_prediction_to_egyptian_pronunciation(w) for w in joined.split())
    )


# Enrich cached glove rows with mapped + TTS text (no gesture model needed here)
import json

with open(GESTURE_CACHE_JSON, encoding="utf-8") as fh:
    _gc = json.load(fh)

for r in _gc.get("real_glove_results", []):
    mapped = map_prediction_to_egyptian_pronunciation(r["predicted"])
    r["mapped"] = mapped
    r["tts_text"] = speak_single_prediction(r["predicted"])

with open(GESTURE_CACHE_JSON, "w", encoding="utf-8") as fh:
    json.dump(_gc, fh, ensure_ascii=False, indent=2)

REAL_GLOVE_RESULTS = _gc.get("real_glove_results", [])
print("Mapping applied to", len(REAL_GLOVE_RESULTS), "glove results")


Mapping applied to 3 glove results


## 5. Load Chatterbox + synthesize

In [5]:
import gc
import inspect
import time
import traceback

try:
    from chatterbox.models.s3gen import S3GEN_SR
except Exception:
    S3GEN_SR = 24000

CHATTER_PATHS: dict[str, str | None] = {}
CHATTER_TIMINGS: dict = {"voice_model": "laghtna_chatterbox_v1", "errs": {}, "items": {}}
CHATTERBOX_SUCCEEDED_SLUGS: set[str] = set()
CHATTERBOX_API = None
CHATTERBOX_LOADED = False
CHATTER_RP_USED = "unsupported"
CHATTER_DIALECT_MSG = ""


def format_bytes(nb: int) -> str:
    if nb < 1024:
        return f"{nb} B"
    if nb < 1024**2:
        return f"{nb / 1024:.2f} KB"
    return f"{nb / 1024 / 1024:.2f} MB"


def unload_cuda(verbose: bool = True):
    gc.collect()
    if torch is not None and CUDA_AVAILABLE:
        torch.cuda.empty_cache()
    if verbose:
        print("[gc+cuda-empty_cache]")


def chatterbox_generate(api_obj, text: str, gen_kw: dict):
    return api_obj.generate(text[:500], **gen_kw)


def load_chatterbox_api():
    global CHATTERBOX_API, CHATTERBOX_LOADED, CHATTER_RP_USED, CHATTER_DIALECT_MSG

    if not (HAVE_CHATTERBOX and HAVE_SAFE and HAVE_HF and torch is not None):
        missing = []
        if not HAVE_CHATTERBOX:
            missing.append("pip install chatterbox-tts")
        if not HAVE_SAFE:
            missing.append("pip install safetensors==0.5.3")
        if not HAVE_HF:
            missing.append("pip install huggingface_hub")
        if torch is None:
            missing.append("pip install torch")
        CHATTER_TIMINGS["errs"]["deps"] = " | ".join(missing)
        print("Chatterbox skip:", CHATTER_TIMINGS["errs"]["deps"])
        return None

    try:
        import sys as _sy_mod

        import perth.perth_net as _pnp

        _pm_root = _sy_mod.modules.get("perth")
        if _pm_root is not None and not callable(getattr(_pm_root, "PerthImplicitWatermarker", None)):
            setattr(_pm_root, "PerthImplicitWatermarker", _pnp.PerthImplicitWatermarker)
    except Exception as _pw:
        print("Perth watermark bridge:", _pw)
        print("Try: pip uninstall -y perth && pip install resemble-perth 'setuptools<81'")

    from huggingface_hub import hf_hub_download
    from safetensors.torch import load_file
    from chatterbox.mtl_tts import ChatterboxMultilingualTTS, Conditionals, SUPPORTED_LANGUAGES
    from chatterbox.models.t3 import T3
    from chatterbox.models.t3.modules.t3_config import T3Config
    from chatterbox.models.s3gen import S3Gen, S3GEN_SR
    from chatterbox.models.voice_encoder import VoiceEncoder
    from chatterbox.models.tokenizers.tokenizer import MTLTokenizer

    try:
        grapheme_path = hf_hub_download(
            repo_id="ResembleAI/chatterbox",
            filename="grapheme_mtl_merged_expanded_v1.json",
            cache_dir=str(CACHE_DIR),
        )
    except Exception:
        CHATTER_TIMINGS["errs"]["hf_tokenizer"] = traceback.format_exc()
        print(traceback.format_exc()[-800:])
        return None

    order = ["cuda", "cpu"] if DEVICE_PREF == "cuda" and CUDA_AVAILABLE else ["cpu"]
    api_obj = None

    for dv in order:
        t0 = time.perf_counter()
        try:
            ck = CHATTERBOX_DIR
            ve = VoiceEncoder()
            ve.load_state_dict(load_file(str(ck / "ve.safetensors")), strict=False)
            ve.eval()
            sg = S3Gen()
            sg.load_state_dict(load_file(str(ck / "s3gen.safetensors")), strict=False)
            t3 = T3(T3Config.multilingual())
            t3.load_state_dict(load_file(str(ck / "t3_mtl23ls_v2.safetensors")), strict=False)
            ve, sg, t3 = ve.to(dv).eval(), sg.to(dv).eval(), t3.to(dv).eval()
            tokzr = MTLTokenizer(grapheme_path)
            cond_path = ck / "conds.pt"
            conds = (
                Conditionals.load(str(cond_path), map_location=torch.device("cpu")).to(dv)
                if cond_path.is_file()
                else None
            )
            api_obj = ChatterboxMultilingualTTS(t3, sg, ve, tokzr, dv, conds=conds)
            CHATTER_TIMINGS["load_s"] = time.perf_counter() - t0
            CHATTER_TIMINGS["device_used"] = dv
            print(f"Chatterbox loaded on {dv} in {CHATTER_TIMINGS['load_s']:.2f}s")
            break
        except torch.cuda.OutOfMemoryError:
            CHATTER_TIMINGS["errs"][f"oom_{dv}"] = "OOM"
            unload_cuda(False)
            api_obj = None
        except Exception:
            CHATTER_TIMINGS["errs"][f"exc_{dv}"] = traceback.format_exc()
            print(traceback.format_exc()[-1200:])
            api_obj = None

    if api_obj is None:
        return None

    _CB_LANGS = SUPPORTED_LANGUAGES
    dialect_hit = sorted({"ar-eg", "arq", "ar_eg", "eg", "egy"} & {str(k).lower() for k in _CB_LANGS})
    if dialect_hit:
        CHATTER_DIALECT_MSG = dialect_hit[0]
        print("Chatterbox selected dialect/voice:", CHATTER_DIALECT_MSG)
    else:
        CHATTER_DIALECT_MSG = "default"
        print("No explicit Egyptian dialect setting found; using model default.")

    arabic_id = next((c for c in ("ar", "arb") if c in _CB_LANGS), "ar")
    print("Chatterbox language_id used:", arabic_id, f"({_CB_LANGS.get(arabic_id, '')})")

    _gen_sig = inspect.signature(api_obj.generate)
    CHATTERBOX_API = api_obj
    CHATTERBOX_LOADED = True
    return api_obj


def synth_chatterbox_phrases(phrases: dict[str, str], repetition_penalty: float = 1.2):
    from IPython.display import Audio as IPA, display

    api_obj = load_chatterbox_api()
    if api_obj is None:
        return

    global CHATTER_RP_USED
    _gen_kw = dict(
        language_id="ar",
        audio_prompt_path=None,
        exaggeration=0.5,
        cfg_weight=0.45,
        temperature=0.75,
    )
    _sig = inspect.signature(api_obj.generate)
    if "repetition_penalty" in _sig.parameters:
        _gen_kw["repetition_penalty"] = repetition_penalty
        CHATTER_RP_USED = str(repetition_penalty)
        print("Chatterbox repetition_penalty used:", CHATTER_RP_USED)
    else:
        CHATTER_RP_USED = "unsupported"
        print("Chatterbox repetition_penalty used: unsupported")

    for slug, text in phrases.items():
        utter = finalize_chatterbox_utterance(text)
        outp = CHATTER_AUDIO_DIR / f"{slug}.wav"
        meta = {"path": str(outp.resolve())}
        try:
            with torch.inference_mode():
                t0 = time.perf_counter()
                wav_tensor = chatterbox_generate(api_obj, utter, _gen_kw)
                meta["inference_s"] = time.perf_counter() - t0
            sr = int(getattr(api_obj, "sr", S3GEN_SR))
            wav_np = wav_tensor.squeeze(0).detach().cpu().numpy().astype("float32")
            wrote = False
            if HAVE_SF:
                import soundfile as sf

                sf.write(str(outp), wav_np, sr, subtype="PCM_16")
                wrote = outp.is_file()
            if not wrote:
                import torchaudio as ta

                w = wav_tensor.squeeze(0).detach().cpu()
                ta.save(str(outp), w.unsqueeze(0) if w.dim() == 1 else w, sr)
                wrote = outp.is_file()
            meta["bytes"] = outp.stat().st_size if wrote else 0
            CHATTER_PATHS[slug] = str(outp.resolve()) if wrote else None
            CHATTER_TIMINGS["items"][slug] = meta
            if wrote:
                CHATTERBOX_SUCCEEDED_SLUGS.add(slug)
            print(
                f"  {slug}: {format_bytes(meta.get('bytes', 0))} | infer {meta.get('inference_s', float('nan')):.3f}s | {utter!r}"
            )
            if wrote:
                display(IPA(filename=str(outp)))
        except Exception:
            CHATTER_TIMINGS["errs"][slug] = traceback.format_exc()
            CHATTER_PATHS[slug] = None
            print(f"  {slug}: FAILED")
            traceback.print_exc()


PHRASES_CB: dict[str, str] = {}

# Demo phrases (letters / numbers / word / sentence)
PHRASES_CB["demo_alef"] = speak_single_prediction("أ")
PHRASES_CB["demo_beh"] = speak_single_prediction("ب")
PHRASES_CB["demo_itneen"] = speak_single_prediction("اثنين")
PHRASES_CB["demo_khamsa"] = speak_single_prediction("خمسة")
PHRASES_CB["demo_letters_abc"] = speak_predictions_as_letters(["أ", "ب", "ت"])
PHRASES_CB["demo_numbers"] = speak_predictions_as_letters(["اثنين", "ثلاثة", "خمسة"])
PHRASES_CB["demo_bayt"] = speak_predictions_as_joined_word(["ب", "ي", "ت"])
PHRASES_CB["demo_mama"] = finalize_chatterbox_utterance(EG_WORD_TTS["ماما"])
PHRASES_CB["demo_sentence"] = finalize_chatterbox_utterance(EG_SENTENCE_TTS["أنا بسمعك"])

for r in REAL_GLOVE_RESULTS:
    PHRASES_CB[r["slug"]] = r.get("tts_text") or speak_single_prediction(r["predicted"])

print("\n--- Phrases for Chatterbox ---")
for k, v in PHRASES_CB.items():
    print(f"  {k}: {v}")

synth_chatterbox_phrases(PHRASES_CB, repetition_penalty=1.2)
unload_cuda()



--- Phrases for Chatterbox ---
  demo_alef: أَلِف.
  demo_beh: بِه.
  demo_itneen: اِتْنِين.
  demo_khamsa: خَمْسَه.
  demo_letters_abc: أَلِف، بِه، تِه.
  demo_numbers: اِتْنِين، تَلَاتَه، خَمْسَه.
  demo_bayt: بَيْت.
  demo_mama: مَامَا.
  demo_sentence: أَنَا بَسْمَعَك.
  real_glove_1: أَلِف.
  real_glove_2: اِتْنِين.
  real_glove_3: خَمْسَه.


C:\Users\sondo\AppData\Local\Programs\Python\Python311\Lib\site-packages\perth\perth_net\__init__.py:1: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


C:\Users\sondo\AppData\Local\Programs\Python\Python311\Lib\site-packages\diffusers\models\lora.py:393: FutureWarning: `LoRACompatibleLinear` is deprecated and will be removed in version 1.0.0. Use of `LoRACompatibleLinear` is deprecated. Please switch to PEFT backend by installing PEFT: `pip install peft`.
  deprecate("LoRACompatibleLinear", "1.0.0", deprecation_message)


C:\Users\sondo\AppData\Local\Programs\Python\Python311\Lib\contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)
The following generation flags are not valid and may be ignored: ['output_attentions']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


loaded PerthNet (Implicit) at step 250,000
Chatterbox loaded on cpu in 14.09s
No explicit Egyptian dialect setting found; using model default.
Chatterbox language_id used: ar (Arabic)
Chatterbox repetition_penalty used: 1.2


...


Sampling:   0%|          | 2/1000 [00:00<02:31,  6.59it/s]


Sampling:   0%|          | 3/1000 [00:00<02:18,  7.22it/s]


Sampling:   0%|          | 3/1000 [00:00<02:26,  6.79it/s]

  real_glove_3: 5.67 KB | infer 8.169s | 'خَمْسَه.'


[gc+cuda-empty_cache]


## 6. Real Gloves Prediction Tests (summary)

In [6]:
print("=" * 60)
print("Real Gloves Prediction Tests")
print("=" * 60)

for r in REAL_GLOVE_RESULTS:
    print("\n" + "-" * 40)
    print(f"Test #{r['test']} ({r.get('name', '')})")
    print("Input numbers:", r["values"])
    print("Expected label:", r.get("expected"))
    print("Predicted label:", r.get("predicted"))
    print("Mapped pronunciation:", r.get("mapped"))
    print("Final text sent to Chatterbox:", r.get("tts_text"))
    p = CHATTER_AUDIO_DIR / f"{r['slug']}.wav"
    print("Audio path:", p.resolve() if p.is_file() else "(missing)")
    if r["slug"] in CHATTER_TIMINGS.get("items", {}):
        print("Inference time (s):", CHATTER_TIMINGS["items"][r["slug"]].get("inference_s"))


Real Gloves Prediction Tests

----------------------------------------
Test #1 (GlovesInput_demo_أ)
Input numbers: [865, 472, 632, 593, 675, -8380, 14150, 410, -610, -1310, 275]
Expected label: أ
Predicted label: أ
Mapped pronunciation: أَلِف
Final text sent to Chatterbox: أَلِف.
Audio path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\chatterbox\real_glove_1.wav
Inference time (s): 10.587112199515104

----------------------------------------
Test #2 (GlovesInput_demo_اثنين)
Input numbers: [780, 400, 645, 782, 946, -8400, 14164, -1900, -100, -1500, -400]
Expected label: اثنين
Predicted label: اثنين
Mapped pronunciation: اِتْنِين
Final text sent to Chatterbox: اِتْنِين.
Audio path: C:\Users\sondo\Desktop\Voice Model Stuff\generated_audio\chatterbox\real_glove_2.wav
Inference time (s): 13.884995799511671

----------------------------------------
Test #3 (GlovesInput_demo_خمسة)
Input numbers: [910, 885, 945, 805, 960, 1700, 16450, 450, 100, -150, -400]
Expected label: خمسة
Pre

## 7. Final verification

In [7]:
_ok_gesture = len(REAL_GLOVE_RESULTS) >= 3 and all(r.get("predicted") for r in REAL_GLOVE_RESULTS)
_ok_cb = CHATTERBOX_LOADED
_ok_map = all(r.get("mapped") and r.get("tts_text", "").endswith(".") for r in REAL_GLOVE_RESULTS)
_need = {r["slug"] for r in REAL_GLOVE_RESULTS} | {
    "demo_alef",
    "demo_bayt",
    "demo_sentence",
}
_ok_audio = _need <= CHATTERBOX_SUCCEEDED_SLUGS and all(
    (CHATTER_AUDIO_DIR / f"{s}.wav").is_file() and (CHATTER_AUDIO_DIR / f"{s}.wav").stat().st_size > 1000
    for s in _need
)
_ok_play = len(CHATTERBOX_SUCCEEDED_SLUGS) > 0

print("\n" + "=" * 52)
print("FINAL VERIFICATION")
print("=" * 52)
print("✅ Gesture predictions executed" if _ok_gesture else "❌ Gesture predictions")
print("✅ Chatterbox loaded" if _ok_cb else "❌ Chatterbox loaded")
print(
    "✅ Egyptian pronunciation mapping working"
    if _ok_map
    else "❌ Egyptian pronunciation mapping"
)
print("✅ Audio generated" if _ok_audio else "❌ Audio generated (see errors above)")
print(
    "✅ Playback widgets displayed"
    if _ok_play
    else "⚠ Playback widgets (run in Jupyter; nbconvert may still save audio files)"
)
print("=" * 52)



FINAL VERIFICATION
✅ Gesture predictions executed
✅ Chatterbox loaded
✅ Egyptian pronunciation mapping working
✅ Audio generated
✅ Playback widgets displayed
